In [1]:
import os
import time
import json
from api_client import TradingDeskAPI
from database import save_snapshots

import re
from dateutil import parser

import re
import calendar
from dateutil import parser

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import PchipInterpolator

from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
from datetime import datetime, timezone
from scipy.interpolate import PchipInterpolator
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import RBFInterpolator

from options import OptionSurface, Deribit, OKX, Bybit

In [2]:
target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

s = OptionSurface()

deribit = Deribit(currencies=["BTC", "ETH"], target_expiry=target_expiry)
okx = OKX(currencies=["BTC", "ETH"], target_expiry=target_expiry)
bybit = Bybit(currencies=["BTC", "ETH"], target_expiry=target_expiry)

s.initialize(currencies=["BTC", "ETH"], exchanges=[deribit, okx, bybit])

spot: 64304.0 volume24h: 4.4692
spot: 1904.1 volume24h: 8.1493
spot: 64306.7 volume24h: 2961.64946357
spot: 1904.78 volume24h: 49983.709999
spot: 64299.4 volume24h: 6079.087642
spot: 1904.72 volume24h: 45058.39833


In [3]:
api = TradingDeskAPI()

markets = api.get_markets()

markets_df = pd.DataFrame(markets)

markets_df.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,clobRewards,oneDayPriceChange,oneHourPriceChange,gameStartTime,secondsDelay,positionIds,gameId,sportsMarketType,eventStartTime,marketMetadata
0,561988,Will Josh Hawley win the 2028 Republican presi...,0x4d4bceee5f59f75230c6438f69fc913a6da7afa4098c...,will-josh-hawley-win-the-2028-republican-presi...,,2028-11-07T00:00:00Z,998076.48261,2025-07-11T19:42:18.131Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2520292,Will the Florida Panthers be named the 2026-27...,0x91bcdadc857d32689f9b27b2483594086fd5f75d92b7...,will-the-florida-panthers-be-named-the-2026-27...,,2027-07-01T03:59:00Z,99801.9074,2026-06-12T19:45:37.786912Z,NaN,NaN,...,"[{'id': '660826', 'conditionId': '0x91bcdadc85...",-0.005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2170255,Bab el-Mandeb Strait effectively closed by Sep...,0x95fc6a4ed7f6856d26aa7f21b9f902d05c9cf76b616f...,bab-el-mandeb-strait-effectively-closed-by-sep...,,2026-04-30T00:00:00Z,99142.4071,2026-05-06T00:27:57.775Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '426137', 'conditionId': '0x95fc6a4ed7...",0.020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1808548,Fed Rate Hike by October 2026 Meeting?,0x059db22dae2d735516017d47d1def0ea43e5d7221259...,fed-rate-hike-by-october-2026-meeting,,2026-12-09T00:00:00Z,99044.6537,2026-03-31T21:37:32.25Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '540830', 'conditionId': '0x059db22dae...",0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1361093,Will Kansas City Chiefs win the 2027 NFL AFC C...,0x983223b53e2dc149079e8919efa1a85f4f2cefe5723e...,will-kansas-city-chiefs-win-the-2027-nfl-afc-c...,,2027-01-25T00:00:00Z,98875.6534,2026-02-09T22:34:40.374442Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '852838', 'conditionId': '0x983223b53e...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "bitcoin price",
    "bitcoin hits",
    "bitcoin above",
    "bitcoin below",
]

def is_btc_market(row):
    text = (str(row["question"]) + " " + str(row.get("description", ""))).lower()

    return any(k in text for k in BTC_KEYWORDS)

btc_df = markets_df[markets_df.apply(is_btc_market, axis=1)]
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,clobRewards,oneDayPriceChange,oneHourPriceChange,gameStartTime,secondsDelay,positionIds,gameId,sportsMarketType,eventStartTime,marketMetadata
8,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98448.05581,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '404092', 'conditionId': '0xac32e73aa9...",0.0005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,3241081,"Will the price of Bitcoin be above $70,000 on ...",0x0dfa3818dbe5f8299ccce7c15c6c6149b1350335a846...,bitcoin-above-70k-on-august-7-2026,NaN,2026-08-07T16:00:00Z,96828.24037,2026-07-31T16:10:27Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,-0.0005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94508.87092,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,-0.0035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,93155.4762,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '493461', 'conditionId': '0x7b9072e6a9...",0.0050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,1339767,"Will Bitcoin dip to $50,000 by December 31, 2026?",0xce3c54c3e773e8b3bf73e4c12af93205b21195d03b9c...,will-bitcoin-dip-to-50000-by-december-31-2026-...,,2027-01-01T05:00:00Z,88718.515,2026-02-05T14:48:04.952Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '527806', 'conditionId': '0xce3c54c3e7...",-0.0200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
for i in btc_df.columns:
    print(i)

id
question
conditionId
slug
resolutionSource
endDate
liquidity
startDate
image
icon
description
outcomes
outcomePrices
volume
active
closed
marketMakerAddress
createdAt
updatedAt
new
featured
submitted_by
archived
resolvedBy
restricted
groupItemTitle
groupItemThreshold
questionID
enableOrderBook
orderPriceMinTickSize
orderMinSize
volumeNum
liquidityNum
endDateIso
startDateIso
hasReviewedDates
volume24hr
volume1wk
volume1mo
volume1yr
clobTokenIds
comboStatus
umaBond
umaReward
volume24hrClob
volume1wkClob
volume1moClob
volume1yrClob
volumeClob
liquidityClob
makerBaseFee
takerBaseFee
customLiveness
acceptingOrders
negRisk
negRiskMarketID
negRiskRequestID
events
ready
funded
acceptingOrdersTimestamp
cyom
competitive
pagerDutyNotificationEnabled
approved
rewardsMinSize
rewardsMaxSpread
spread
oneWeekPriceChange
oneMonthPriceChange
oneYearPriceChange
lastTradePrice
bestBid
bestAsk
automaticallyActive
clearBookOnStart
seriesColor
showGmpSeries
showGmpOutcome
manualActivation
negRiskOther
uma

In [6]:
btc_df = btc_df[
            (btc_df["acceptingOrders"] == True) &
            (btc_df["enableOrderBook"] == True)
        ]

btc_df["tokens"] = btc_df["clobTokenIds"].apply(json.loads)
btc_df["yes_token"] = btc_df["tokens"].apply(lambda x: x[0])
btc_df["no_token"] = btc_df["tokens"].apply(lambda x: x[1])

btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,gameStartTime,secondsDelay,positionIds,gameId,sportsMarketType,eventStartTime,marketMetadata,tokens,yes_token,no_token
8,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98448.05581,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[613689431282552874145652703368566154530006753...,6136894312825528741456527033685661545300067537...,9699347185440015640867052761315094444335927219...
18,3241081,"Will the price of Bitcoin be above $70,000 on ...",0x0dfa3818dbe5f8299ccce7c15c6c6149b1350335a846...,bitcoin-above-70k-on-august-7-2026,NaN,2026-08-07T16:00:00Z,96828.24037,2026-07-31T16:10:27Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[379279634038544742932244019214374299379054081...,3792796340385447429322440192143742993790540817...,1140163690003052416621136255380722881681675618...
36,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94508.87092,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[596839742467995258312442199211144557362757050...,5968397424679952583124421992111445573627570503...,7012278860989242444626087945307690553239457130...
47,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,93155.4762,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[955325244079599031455105836849790497350285024...,9553252440795990314551058368497904973502850245...,1078699501271841417437724297387015787237705826...
91,1339767,"Will Bitcoin dip to $50,000 by December 31, 2026?",0xce3c54c3e773e8b3bf73e4c12af93205b21195d03b9c...,will-bitcoin-dip-to-50000-by-december-31-2026-...,,2027-01-01T05:00:00Z,88718.515,2026-02-05T14:48:04.952Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[138879835511298629926051230887441759270844356...,1388798355112986299260512308874417592708443563...,4183266428584583245903064096989148366388472169...


In [7]:
def get_books(row):

    yes_book = api.get_orderbook(row.yes_token)
    no_book = api.get_orderbook(row.no_token)

    return pd.Series({
        "yes_book": yes_book,
        "no_book": no_book,

        "yes_bid": float(yes_book["bids"][-1]["price"]) if yes_book["bids"] else None,
        "yes_ask": float(yes_book["asks"][-1]["price"]) if yes_book["asks"] else None,

        "no_bid": float(no_book["bids"][-1]["price"]) if no_book["bids"] else None,
        "no_ask": float(no_book["asks"][-1]["price"]) if no_book["asks"] else None,
    })

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(get_books, [row for _, row in btc_df.iterrows()]))

btc_df[
    [
        "yes_book",
        "no_book",
        "yes_bid",
        "yes_ask",
        "no_bid",
        "no_ask"
    ]
] = pd.DataFrame(results).values

In [8]:
def extract_strike(question):

    match = re.search(
        r'\$([\d,]+)',
        question
    )

    if match:
        return float(
            match.group(1).replace(",", "")
        )

    return None

def extract_expiry(question, slug=None):
    # Try full date in question
    # m = match
    m = re.search(
        r'(January|February|March|April|May|June|July|August|September|October|November|December) \d{1,2}, \d{4}',
        question
    )
    if m:
        return parser.parse(m.group()).strftime("%Y%m%d")

    if slug:
        slug = slug.lower()

        # Full date in slug: december-31-2026
        m = re.search(
            r'(january|february|march|april|may|june|july|august|september|october|november|december)-(\d{1,2})-(\d{4})',
            slug
        )
        if m:
            month, day, year = m.groups()
            return parser.parse(f"{day} {month} {year}").strftime("%Y%m%d")

        # Month only: august-2026
        m = re.search(
            r'(january|february|march|april|may|june|july|august|september|october|november|december)-(\d{4})',
            slug
        )
        if m:
            month_name, year = m.groups()
            month = parser.parse(month_name).month
            year = int(year)

            last_day = calendar.monthrange(year, month)[1]
            return f"{year}{month:02d}{last_day:02d}"

    return None

btc_df["strike"] = btc_df.question.apply(
    extract_strike
)

btc_df["expiry"] = btc_df.apply(
    lambda row: extract_expiry(row["question"], row["slug"]),
    axis=1,
)

In [9]:
def parse_btc_market_type(question):

    q = question.lower()

    # ----------------------
    # Direction
    # ----------------------
    if any(word in q for word in [
        "dip",
        "fall",
        "drop",
        "below",
        "under",
        "crash"
    ]):
        direction = "down"

    elif any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "above",
        "over",
        "exceed"
    ]):
        direction = "up"

    else:
        direction = None

    # ----------------------
    # Event type
    # ----------------------
    if any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "dip to",
        "fall to",
        "drop to",
        "crash to"
    ]):
        event_type = "touch"

    elif any(word in q for word in [
        "be above",
        "above on",
        "close above",
        "be below",
        "below on",
        "close below",
        "finish above",
        "finish below"
    ]):
        event_type = "expiry"


    else:
        event_type = None

    return {
        "direction": direction,
        "event_type": event_type
    }

btc_df[
    ["direction", "event_type"]
] = btc_df["question"].apply(
    lambda x: pd.Series(parse_btc_market_type(x))
)

In [12]:
btc_df["expiry_dt"] = pd.to_datetime(btc_df["expiry"], format="%Y%m%d", utc=True)
btc_df["T"] = (btc_df["expiry_dt"] - datetime.now(timezone.utc)).dt.total_seconds() / (365.25 * 24 * 3600)
btc_df = btc_df[btc_df["T"] > 0]

In [13]:
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,yes_bid,yes_ask,no_bid,no_ask,strike,expiry,direction,event_type,expiry_dt,T
8,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98448.05581,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.021,0.023,0.977,0.979,200000.0,20261231,up,touch,2026-12-31 00:00:00+00:00,0.398918
36,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94508.87092,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.032,0.039,0.961,0.968,20000.0,20261231,down,touch,2026-12-31 00:00:00+00:00,0.398918
47,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,93155.4762,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.71,0.72,0.28,0.29,70000.0,20261231,up,touch,2026-12-31 00:00:00+00:00,0.398918
91,1339767,"Will Bitcoin dip to $50,000 by December 31, 2026?",0xce3c54c3e773e8b3bf73e4c12af93205b21195d03b9c...,will-bitcoin-dip-to-50000-by-december-31-2026-...,,2027-01-01T05:00:00Z,88718.515,2026-02-05T14:48:04.952Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.34,0.35,0.65,0.66,50000.0,20261231,down,touch,2026-12-31 00:00:00+00:00,0.398918


In [41]:
def calculate_ev(
        model_prob,
        best_ask_yes,
        best_bid_yes,
        best_ask_no,
        best_bid_no,
        fee_rate=0.07,
        kelly_fraction = 0.5
    ):
    """
    Expected PnL per contract for Polymarket.

    model_prob : P(YES)
    fee_rate   : taker fee rate (e.g. 0.07 for crypto markets)

    Assumes:
      - you are a taker
      - fee = fee_rate * price * (1 - price)
      - settlement has no fee
    """

    def is_valid(price):
        return pd.notna(price)

    p_yes = model_prob
    p_no = 1 - p_yes

    def fee(price):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    # -------------------------
    # BUY YES
    # -------------------------
    if is_valid(best_ask_yes):
        buy_yes_cost = best_ask_yes + fee(best_ask_yes)

        profit_if_yes = 1 - buy_yes_cost
        cost_if_no = buy_yes_cost

        # EV: profit if yes - cost if no
        buy_yes_ev = p_yes * profit_if_yes - p_no * cost_if_no

        """
        p            : probability of winning
        win_profit   : net profit if the bet wins
        loss_amount  : net loss if the bet loses

        Returns the optimal Kelly fraction.
        """

        buy_yes_kelly = kelly_fraction * (buy_yes_ev / profit_if_yes) # fee adjusted kelly

        # -------------------------
        # SELL YES (short YES)
        # -------------------------
        sell_yes_credit = best_bid_yes - fee(best_bid_yes)

        profit_if_no = sell_yes_credit
        cost_if_yes = 1 - sell_yes_credit # need to pay the remaining of the $1 out of the credit you got, if yes happened

        # EV: profit if yes - cost if no
        sell_yes_ev = p_no * profit_if_no - p_yes * cost_if_yes

        sell_yes_kelly = kelly_fraction * (sell_yes_ev / profit_if_no)

    else:
        buy_yes_ev = np.nan
        sell_yes_ev = np.nan
        buy_yes_kelly = np.nan
        sell_yes_kelly = np.nan
        
    # -------------------------
    # BUY NO
    # -------------------------
    if is_valid(best_ask_no):
        buy_no_cost = best_ask_no + fee(best_ask_no)

        profit_if_no = 1 - buy_no_cost
        cost_if_yes = buy_no_cost

        buy_no_ev = p_no * profit_if_no - p_yes * cost_if_yes

        buy_no_kelly = kelly_fraction * (buy_no_ev / profit_if_no)

        # -------------------------
        # SELL NO (short NO)
        # -------------------------
        sell_no_credit = best_bid_no - fee(best_bid_no)

        profit_if_yes = sell_no_credit
        cost_if_no = 1 - sell_no_credit

        sell_no_ev = p_yes * profit_if_yes - p_no * cost_if_no

        sell_no_kelly = kelly_fraction * (sell_no_ev / profit_if_yes)

    else:
        buy_no_ev = np.nan
        sell_no_ev = np.nan
        buy_no_kelly = np.nan
        sell_no_kelly = np.nan

    print("buy_yes_ev:", buy_yes_ev)
    print("sell_yes_ev:", sell_yes_ev)
    print("buy_no_ev:", buy_no_ev)
    print("sell_no_ev:", sell_no_ev)
    print("buy_yes_kelly:", buy_yes_kelly)
    print("sell_yes_kelly:", sell_yes_kelly)
    print("buy_no_kelly:", buy_no_kelly)
    print("sell_no_kelly:", sell_no_kelly)

    return {
        "buy_yes_fee": fee(best_ask_yes),
        "sell_yes_fee": fee(best_bid_yes),
        "buy_no_fee": fee(best_ask_no),
        "sell_no_fee": fee(best_bid_no),
        "buy_yes_ev": buy_yes_ev,
        "sell_yes_ev": sell_yes_ev,
        "buy_no_ev": buy_no_ev,
        "sell_no_ev": sell_no_ev,
        "buy_yes_kelly": buy_yes_kelly,
        "sell_yes_kelly": sell_yes_kelly,
        "buy_no_kelly": buy_no_kelly,
        "sell_no_kelly": sell_no_kelly,
    }


result = calculate_ev(
    model_prob=0.059,
    best_ask_yes=0.1,
    best_bid_yes=0.09,
    best_ask_no=0.91,
    best_bid_no=0.9,
    fee_rate=0.07
)

buy_yes_ev: -0.04730000000000002
sell_yes_ev: 0.025266999999999998
buy_no_ev: 0.025266999999999984
sell_no_ev: -0.04729999999999996
buy_yes_kelly: -0.026463018910148833
sell_yes_kelly: 0.14992227087709303
buy_no_kelly: 0.14992227087709298
sell_no_kelly: -0.026463018910148794


In [42]:
def calculate_market_ev(row, s, confidence=0.7):

    currency = "BTC"
    required_strike = row["strike"]

    T = row["T"]

    iv = s.get_iv_from_surface(exchange=s, currency=currency, required_strike=required_strike, T=T)
    print('iv', iv)

    if iv is None:
        return None

    if row["event_type"] == "touch":

        p_touch_above, p_touch_below = s.prob_touch(
                    spot=s.data[currency]["weighted_spot"],
                    required_strike=required_strike,
                    iv=iv,
                    T=T,
                    r=0)
        
        if row["direction"] == "up":
            model_prob = p_touch_above

        elif row["direction"] == "down":
            model_prob = p_touch_below

    if row["event_type"] == "expiry":

        p_finish_above, p_finish_below = s.prob_finish(
                    spot=s.data[currency]["weighted_spot"], 
                    required_strike=required_strike, 
                    iv=iv,
                    T=T,
                    r=0)

        if row["direction"] == "up":
            model_prob = p_finish_above

        elif row["direction"] == "down":
            model_prob = p_finish_below

    market_prob = (
        row["yes_bid"] + row["yes_ask"]
    ) / 2

    model_prob_adjusted = (
        confidence * model_prob
        +
        (1-confidence) * market_prob
    )

    ev = calculate_ev(
        model_prob=model_prob,
        best_ask_yes=row["yes_ask"],
        best_bid_yes=row["yes_bid"],
        best_ask_no=row["no_ask"],
        best_bid_no=row["no_bid"],
    )

    return pd.Series({
        "iv": iv,
        "model_prob": model_prob,
        **ev
    })

btc_df[["iv", "model_prob", "buy_yes_ev", "sell_yes_ev", "buy_no_ev", "sell_no_ev",
        "buy_yes_fee", "sell_yes_fee", "buy_no_fee", "sell_no_fee",
        "buy_yes_kelly", "sell_yes_kelly", "buy_no_kelly", "sell_no_kelly"]] = btc_df.apply(
    lambda row: calculate_market_ev(row, s),
    axis=1,
)

required_strike: 200000.0 T: 0.39891835750126753
iv 0.559774351375171
p_touch_above: 0.00067 p_touch_below: 1.0
buy_yes_ev: -0.023902970000000003
sell_yes_ev: 0.018890870000000004
buy_no_ev: 0.018890870000000066
sell_no_ev: -0.02390297000000003
buy_yes_kelly: -0.012252566960339413
sell_yes_kelly: 0.48287397237443946
buy_no_kelly: 0.4828739723744395
sell_no_kelly: -0.012252566960339427
required_strike: 20000.0 T: 0.39891835750126753
iv 1.119548702750342
p_touch_above: 1.0 p_touch_below: 0.1594
buy_yes_ev: 0.11777647
sell_yes_ev: -0.12956832
buy_no_ev: -0.12956831999999996
sell_no_ev: 0.11777646999999991
buy_yes_kelly: 0.0614458272332166
sell_yes_kelly: -2.171656440401613
buy_no_kelly: -2.1716564404016108
sell_no_kelly: 0.061445827233216566
required_strike: 70000.0 T: 0.39891835750126753
iv 1.119548702750342
p_touch_above: 0.82966 p_touch_below: 0.99973
buy_yes_ev: 0.09554799999999997
sell_yes_ev: -0.134073
buy_no_ev: -0.13407299999999994
sell_no_ev: 0.09554799999999997
buy_yes_kelly: 0.

In [43]:
btc_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,best_ev,best_action,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly
8,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98448.05581,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.023903,0.018891,0.018891,-0.023903,0.001573,sell_no_ev,-0.012253,0.482874,0.482874,-0.012253
36,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94508.87092,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.117776,-0.129568,-0.129568,0.117776,0.002624,sell_no_ev,0.061446,-2.171656,-2.171656,0.061446
47,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,93155.4762,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.095548,-0.134073,-0.134073,0.095548,0.014413,sell_yes_ev,0.179677,-0.096374,-0.096374,0.179677
91,1339767,"Will Bitcoin dip to $50,000 by December 31, 2026?",0xce3c54c3e773e8b3bf73e4c12af93205b21195d03b9c...,will-bitcoin-dip-to-50000-by-december-31-2026-...,,2027-01-01T05:00:00Z,88718.515,2026-02-05T14:48:04.952Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.150795,-0.192428,-0.192428,0.150795,0.015925,buy_yes_ev,0.118909,-0.296689,-0.296689,0.118909


In [44]:
btc_df = btc_df.dropna(subset=["buy_yes_ev"])

ev_cols = [
    "buy_yes_ev",
    "sell_yes_ev",
    "buy_no_ev",
    "sell_no_ev"
]

btc_df["best_ev"] = btc_df[ev_cols].max(axis=1)

btc_df["best_action"] = btc_df[ev_cols].idxmax(axis=1)

In [53]:
for i in opportunities.columns:
    print(i)

id
question
conditionId
slug
resolutionSource
endDate
liquidity
startDate
image
icon
description
outcomes
outcomePrices
volume
active
closed
marketMakerAddress
createdAt
updatedAt
new
featured
submitted_by
archived
resolvedBy
restricted
groupItemTitle
groupItemThreshold
questionID
enableOrderBook
orderPriceMinTickSize
orderMinSize
volumeNum
liquidityNum
endDateIso
startDateIso
hasReviewedDates
volume24hr
volume1wk
volume1mo
volume1yr
clobTokenIds
comboStatus
umaBond
umaReward
volume24hrClob
volume1wkClob
volume1moClob
volume1yrClob
volumeClob
liquidityClob
makerBaseFee
takerBaseFee
customLiveness
acceptingOrders
negRisk
negRiskMarketID
negRiskRequestID
events
ready
funded
acceptingOrdersTimestamp
cyom
competitive
pagerDutyNotificationEnabled
approved
rewardsMinSize
rewardsMaxSpread
spread
oneWeekPriceChange
oneMonthPriceChange
oneYearPriceChange
lastTradePrice
bestBid
bestAsk
automaticallyActive
clearBookOnStart
seriesColor
showGmpSeries
showGmpOutcome
manualActivation
negRiskOther
uma

In [48]:
opportunities = btc_df[btc_df["best_ev"] > 0].sort_values("best_ev", ascending=False)

opportunities

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,best_ev,best_action,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly
91,1339767,"Will Bitcoin dip to $50,000 by December 31, 2026?",0xce3c54c3e773e8b3bf73e4c12af93205b21195d03b9c...,will-bitcoin-dip-to-50000-by-december-31-2026-...,,2027-01-01T05:00:00Z,88718.515,2026-02-05T14:48:04.952Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.150795,-0.192428,-0.192428,0.150795,0.015925,buy_yes_ev,0.118909,-0.296689,-0.296689,0.118909
47,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,93155.4762,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.095548,-0.134073,-0.134073,0.095548,0.014413,sell_yes_ev,0.179677,-0.096374,-0.096374,0.179677
36,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94508.87092,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.117776,-0.129568,-0.129568,0.117776,0.002624,sell_no_ev,0.061446,-2.171656,-2.171656,0.061446
8,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,98448.05581,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.023903,0.018891,0.018891,-0.023903,0.001573,sell_no_ev,-0.012253,0.482874,0.482874,-0.012253


In [ ]:
MAX_POSITION = 0.05



In [ ]:
for _, trade in opportunities.iterrows():
    print(trade)

1339767
2467210
1343219
701486


In [ ]:
fills = pd.read_parquet("fills.parquet")

buys = fills[fills.action == "BUY"]
sells = fills[fills.action == "SELL"]

position = (
    buys.shares.sum()
    -
    sells.shares.sum()
)

EXIT_THRESHOLD = -0.02

def get_exit_ev(pos, row):

    if pos["outcome"].lower() == "yes":
        return row["sell_yes_ev"]

    elif pos["outcome"].lower() == "no":
        return row["sell_no_ev"]
    
market_lookup = btc_df.set_index("conditionId")

positions = api.get_positions()

for pos in positions:

    if pos["condition_id"] not in market_lookup.index:
        continue

    row = market_lookup.loc[pos["condition_id"]]

    if pos.outcome == "Yes":
        hold_ev = row.buy_yes_ev
        exit_ev = row.sell_yes_ev

    elif pos.outcome == "No":
        hold_ev = row.buy_no_ev
        exit_ev = row.sell_no_ev

    if hold_ev < EXIT_THRESHOLD:
        api.close_position(pos)

    entry_price = get_entry_price(pos)

    pnl = (price - entry_price) * size

    print(
        pos["outcome"],
        pos["shares"],
        row["question"],
        hold_ev,
        exit_ev
    )



In [ ]:
MIN_EV = 0.01   # require 1% edge, default = 0
bankroll = 100
MAX_POSITION = 0.05

positions = []

for _, trade in opportunities.iterrows():

    if trade.best_ev < MIN_EV:
        continue

    if trade.best_action == "buy_yes_ev":

        token = trade.yes_token
        side = "YES"
        kelly = trade.buy_yes_kelly
        price = trade.yes_ask

    elif trade.best_action == "sell_yes_ev": # usually cannot short, this only happens when im closing positions

        token = trade.yes_token
        side = "YES"
        kelly = trade.sell_yes_kelly
        price = trade.yes_bid

    elif trade.best_action == "buy_no_ev":

        token = trade.no_token
        side = "NO"
        kelly = trade.buy_no_kelly
        price = trade.no_ask

    elif trade.best_action == "sell_no_ev":

        token = trade.no_token
        side = "NO"
        kelly = trade.sell_no_kelly
        price = trade.no_bid

    dollars = bankroll * kelly

    dollars = min(dollars, bankroll * MAX_POSITION)

    position = api.get_conditional_balance(
        token
    )

    current_shares = float(
        position["balance"]
    )

    # inventory cap
    dollars = min(dollars, bankroll * MAX_POSITION - current_shares * price)

    if dollars <= 0:
        continue

    shares = dollars / price

    api.place_limit_order(
        side=side,
        shares=shares,
        price=price
    )

    avg_entry_price =
        (old_shares * old_price + new_shares * new_price)
        /
        (total_shares)

    fills_df.append({
        "condition_id": trade.conditionId,
        "token_id": token_id,
        "outcome": side,
        "size": shares,
        "entry_price": price,
        "entry_time": datetime.now()
    })




In [ ]:
def save(df):

    date = datetime.now().strftime("%Y%m%d")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    DATA_DIR = f"data/{date}"
    os.makedirs(DATA_DIR, exist_ok=True)
    filename = f"{DATA_DIR}/markets_{timestamp}.parquet"

    df.to_parquet(filename, engine="fastparquet", index=False)

    print('df saved at: ', filename)

save(btc_df)

In [ ]:
ENTRY_THRESHOLD = 0.03
EXIT_THRESHOLD = 0.01

edge = 0.031

if edge > ENTRY_THRESHOLD:
    pass
    #sell_yes()

if edge < EXIT_THRESHOLD:
    pass
    #close_position()

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
while True:

    # 1. Manage inventory
    positions = api.get_positions()

    for pos in positions:

        market = model.get_market(pos.condition_id)

        if pos.outcome == "Yes":
            hold_ev = market.buy_yes_ev
            exit_ev = market.sell_yes_ev

        else:
            hold_ev = market.buy_no_ev
            exit_ev = market.sell_no_ev

        if hold_ev < EXIT_THRESHOLD:
            api.close_position(pos)


    # 2. Find new opportunities
    opportunities = scan_markets()

    for trade in opportunities:

        if trade.best_ev > MIN_EV:
            api.place_limit_order(...)


In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]